# 2.1 算子执行流程

本节详细介绍自定义算子的完整执行流程，包括 aclnn 单算子 API 调用和 GE 图模式两种主要调用方式的执行流程。

---

## 学习目标

完成本节后，你将能够：
- 理解算子的执行流程架构
- 掌握 aclnn 单算子 API 调用流程
- 掌握 GE 图模式执行流程
- 了解 内核调用符<<<>>> kernel直调的执行方式
- 掌握各阶段的调试方法

---

## 1. 环境准备
正式开始学习之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成算子的开发及编译。

In [ ]:
!mkdir -p Sources/02.02
!mkdir -p Sources/02.02/add

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))

---

## 2. 算子执行通路介绍

算子开发完成后，需要验证完整的通路，需对算子的执行过程有所了解，下图介绍了框架到算子的执行流程，共5个通路。

<img src="./images/算子执行通路.png" alt="算子执行通路"  width="700px" >

从上层框架到算子执行，一共有5条路径，开发过程中，重点关注的是路径3和路径5，也是本节课程重点介绍的路径。


路径3：即所谓的图模式。图模式是相对于单算子模式的，图模式下，通过有向无环图，描述网络的计算逻辑。图模式的Host调度，可以避免总是返回Python调用栈，避免冗余流程与数据结构转换，并且可以直接使用图编译阶段完成的Infer Shape与Tiling计算结果。

路径5：即所谓的单算子模式或者eager模式（即时执行），由Host CPU逐个下发算子，一个算子的下发流程包含Python处理、Python到C++数据结构转换、Tiling计算、申请算子的Workspace内存和输出内存、Launch等Host操作。为了加速单算子Host调度，在Pytorch中，昇腾适配层采用了生产者-消费者双线程模式加速，生产者线程主要负责Launch之前的处理动作，消费者线程主要负责Launch算子，对标路径2。


Ascend C算子常见的调用方式有kernel直调、单算子调用、第三方框架中调用。工程化的算子通常以单算子调用和第三方框架中调用为主。具体调用方式和调用条件如下：

<table >
    <tr>
        <th style="border: 1px solid #ddd; padding: 12px; text-align: left; font-weight: bold; min-width: 180px;">调用方式</th>
        <th style="border: 1px solid #ddd; padding: 12px; text-align: left; font-weight: bold; min-width: 180px;">使用条件</th>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">单算子API</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">单算子模型执行</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子入图开发，算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">IR构图</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子入图开发，算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">Pytorch框架调用</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">插件适配开发，算子工程编译部署</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">TensorFlow框架调用</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">插件适配开发，算子工程编译部署</td>
    </tr>
        <tr>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">Pybind调用</td>
        <td style="border: 1px solid #ddd; padding: 10px; vertical-align: middle; text-align: left;">算子工程编译部署</td>
    </tr>
</table>  

---

## 3. 单算子调用执行流程

下面我们以Add算子为例，基于torch框架，介绍单算子API的执行流程。

Add算子的介绍参见CANN社区：https://gitcode.com/cann/ops-math/blob/master/math/add/README.md



step1: 用户使用torch，调用Add算子。

In [ ]:
import torch
import torch_npu

x = torch.randint(-5, 5, (2, 2), dtype=torch.int32).npu()
y = torch.randint(-5, 5, (2, 2), dtype=torch.int32).npu()

z = torch.add(x, y)

step2：torch层调用Add算子的aclnn接口，aclnn分为两段式接口，会先调用aclnnAddGetWorkspace分配内存，然后调用aclnnAdd接口。

step3: aclnn根据注册的算子原型，找到算子执行的二进制，即kernel.o文件。

step4: 进入Add算子的Tiling计算阶段，完成相关tilingData计算。

step5: rts 进行kernelLaunch调度执行

step6: Add算子kernel，从GM上CopyIn数据，并根据TilingData，在AICore上执行Compute计算，最后从UB上CopyOut到GM上。

详细流程如图：

<img src="./images/Add算子执行流程.png" alt="Add算子执行接口流程"  width="700px" >


---

## 4. 图模式调用执行流程


### 4.1 静态图执行流程

静态图，即图上算子Shape在编译时是已知且明确的（shape为[10, 2]）。

以Model粒度下发执行：

- 模型加载时GE向rts申请模型流，按顺序下发所有算子的Kernel，此时模型不会立即执行

- 模型执行开始时，由GE调用rts的rtModelExecute接口下发一个ModelExecute任务，触发流上的tasks启动，执行过程中没有host调用

静态图执行示意：

<img src="./images/静态图执行流程.png" alt="静态图执行示意"  width="700px" >


### 4.2 动态图执行流程

动态图：图上算子Shape在编译时未知（shape中包含-1或-2），编译结果需要支持（range内）所有shape的执行。

动态图分类：

- 纯动态图 —— 网络输入时动态

- 动静混合 —— 网络中部分输入动态，部分输入静态；或者网络输入为静态，但存在二三类算子导致inferShape推导出动态。

动态图执行：

走GE的动态图执行器（常说的runtime2.0）按照拓扑序调用每个算子的inferShape、Tiling，为每个算子分配输出内存和WorkSpace内存，组装args后调用rtKenelLaunch完成算子下发。


动态图执行示意：

<img src="./images/动态图执行流程.png" alt="动态图执行示意"  width="700px" >


---

## 5、使用内核调用符<<<>>> kernel直调执行流程

以Add算子为例子，演示内核调用符<<<>>> kernel直调执行流程。

- 样例功能：  
  Add样例实现了两个数据相加，返回相加结果的功能。对应的数学表达式为：  
  ```
  z = x + y
  ```

- 样例规格：
  <table>
  <tr><td rowspan="1" align="center">样例类型(OpType)</td><td colspan="4" align="center">Add</td></tr>
  <tr><td rowspan="3" align="center">样例输入</td><td align="center">name</td><td align="center">shape</td><td align="center">data type</td><td align="center">format</td></tr>
  <tr><td align="center">x</td><td align="center">[8, 2048]</td><td align="center">float</td><td align="center">ND</td></tr>
  <tr><td align="center">y</td><td align="center">[8, 2048]</td><td align="center">float</td><td align="center">ND</td></tr>
  <tr><td rowspan="1" align="center">样例输出</td><td align="center">z</td><td align="center">[8, 2048]</td><td align="center">float</td><td align="center">ND</td></tr>
  <tr><td rowspan="1" align="center">核函数名</td><td colspan="4" align="center">add_custom</td></tr>
  </table>

- 样例实现：
  - Kernel实现  
    本样例使用8个核完成计算，每个核处理2048个元素。计算偏移量为：
    ```
    block_idx * blockLength
    ```

    Add样例的实现流程分为3个步骤：

    **第一步：搬运数据到UB（Unified Buffer）**
    
    将GM（Global Memory）上的输入x和y搬运到UB（Unified Buffer）上的xLocal、yLocal中。
    
    **第二步：执行向量加法**
    
    对xLocal、yLocal执行加法操作，计算结果存储在UB（Unified Buffer）上的zLocal中。
    
    **第三步：搬运结果到GM（Global Memory）**
    
    将输出数据从zLocal搬运至GM（Global Memory）上的输出z中。

- 调用实现  
  使用内核调用符<<<>>>调用核函数。

样例代码如下：

In [ ]:
%%writefile Sources/02.02/add/Add_kernel_call_test.cpp

#include <cstdint>
#include <vector>
#include <algorithm>
#include <iterator>

#include <iostream>

#include "acl/acl.h"

#include "kernel_operator.h"

template <uint32_t blockLength>
__vector__ __global__ void add_custom(__gm__ float* x, __gm__ float* y, __gm__ float* z)
{
    AscendC::InitSocState();

    AscendC::GlobalTensor<float> xGm, yGm, zGm;
    xGm.SetGlobalBuffer(x + block_idx * blockLength, blockLength);
    yGm.SetGlobalBuffer(y + block_idx * blockLength, blockLength);
    zGm.SetGlobalBuffer(z + block_idx * blockLength, blockLength);

    AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    AscendC::LocalTensor<float> xLocal = ubAllocator.Alloc<float, blockLength>();
    AscendC::LocalTensor<float> yLocal = ubAllocator.Alloc<float, blockLength>();
    AscendC::LocalTensor<float> zLocal = ubAllocator.Alloc<float, blockLength>();

    AscendC::DataCopy(xLocal, xGm, blockLength);
    AscendC::DataCopy(yLocal, yGm, blockLength);
    AscendC::PipeBarrier<PIPE_ALL>();

    AscendC::Add(zLocal, xLocal, yLocal, blockLength);
    AscendC::PipeBarrier<PIPE_ALL>();

#if 0
    // Debug I/O. Set #if 1 to enable print.
    AscendC::printf("%s\n", "[ DumpTensor in xLocal]");
    AscendC::DumpTensor(xLocal, 1, 32);

    AscendC::printf("%s\n", "[ DumpTensor in yLocal]");
    AscendC::DumpTensor(yLocal, 2, 32);

    AscendC::printf("%s\n", "[ DumpTensor in zLocal]");
    AscendC::DumpTensor(zLocal, 3, 32);
#endif

    AscendC::DataCopy(zGm, zLocal, blockLength);
    AscendC::PipeBarrier<PIPE_ALL>();
}

std::vector<float> kernel_add(std::vector<float>& x, std::vector<float>& y)
{
    constexpr uint32_t numBlocks = 8;
    constexpr uint32_t blockLength = 2048;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    float* xDevice = nullptr;
    float* yDevice = nullptr;
    float* zDevice = nullptr;
    uint8_t* zHost = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);

    aclrtMalloc((void**)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMallocHost((void**)&zHost, totalByteSize);

    aclrtMemcpy(xDevice, totalByteSize, x.data(), totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, y.data(), totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);

    add_custom<blockLength><<<numBlocks, 0>>>(xDevice, yDevice, zDevice);
    aclrtSynchronizeDevice();

    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z((float*)zHost, (float*)(zHost + totalByteSize));

    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);

    aclrtResetDevice(deviceId);
    aclFinalize();

    return z;
}

uint32_t VerifyResult(std::vector<float>& output, std::vector<float>& golden)
{
    auto printTensor = [](std::vector<float>& tensor, const char* name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(
            tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "test pass!" << std::endl;
        return 0;
    } else {
        std::cout << "test failed!" << std::endl;
        return 1;
    }
    return 0;
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    std::vector<float> x(totalLength);
    std::vector<float> y(totalLength);

    for (uint32_t i = 0; i < totalLength; ++i) {
        x[i] = i * 0.1f;
        y[i] = i * 0.2f;
    }

    std::vector<float> output = kernel_add(x, y);

    std::vector<float> golden(totalLength);
    for (uint32_t i = 0; i < totalLength; ++i) {
        golden[i] = x[i] + y[i];
    }

    return VerifyResult(output, golden);
}

编写CMakeLists.txt

In [ ]:
%%writefile Sources/02.02/add/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture: dav-2201, dav-3510")

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    Add_kernel_call_test.cpp
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

样例执行：

In [45]:
# 配置环境变量
!source $ASCEND_TOOLKIT_HOME/set_env.sh

In [ ]:
!mkdir -p Sources/02.02/add/build      # 创建并进入build目录
!cd Sources/02.02/add/build && cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 ..; make -j    # 编译工程，默认npu模式
!./demo                           # 执行样例

---

## 课后练习

完成以下题目，检验你对算子执行流程的理解：

1. （判断题）aclnn 单算子 API 调用需要先完成算子工程的编译部署。    

2. （判断题）静态图模式下，算子的 Shape 在编译时是已知且明确的，动态图模式下 Shape 在编译时未知。    

3. （单选题）aclnn 接口采用两段式调用，先调用哪个接口分配内存？  
    A. aclnnAdd  
    B. aclnnAddGetWorkspace  
    C. aclrtMalloc  
    D. aclrtMemcpy  

4. （单选题）内核调用符 <<<>>> 的语法中，threadsPerBlock 参数表示什么？  
    A. 线程块的个数  
    B. 每个线程块内的线程数量  
    C. 动态申请内存大小  
    D. 流同步标识  

5. （单选题）动态图执行流程中，按照拓扑序调用每个算子的哪些操作？  
    A. inferShape 和 Tiling  
    B. 编译和执行  
    C. 数据生成和验证  
    D. 内存分配和释放  

**执行以下代码获取答案。**

---

上一节：[2.0 算子加速库基础章节导引](./02.00_chapter_intro.ipynb)

In [ ]:
!cat ./answer/02.01_answer.txt